<a href="https://colab.research.google.com/github/crypt-asu/Ascon-Implementation/blob/main/PeerJ_CKKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Code DesiloFHE**

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import time
from desilofhe import Engine

# ─────────────────────────────────────────────────────────────────
def chebyshev_coeffs_step(degree: int, delta: float = 0.05):
    """
    Step fonksiyonu için Chebyshev katsayılarını hesaplar.
    step(x) = 1 if x > 0 else 0
    """
    from scipy.fftpack import dct

    # Grid points
    n = degree + 1
    x = np.cos(np.pi * (np.arange(n) + 0.5) / n)

    # Target function (step at 0)
    # x in [-1, 1]
    y = np.where(x > 0, 1.0, 0.0)

    # DCT to find coefficients
    c = dct(y, type=2, norm='ortho')
    # Normalize for standard Chebyshev definition
    c[0] /= np.sqrt(2)
    c *= (np.sqrt(2.0 / n))
    return c
def eval_chebyshev_plaintext(coeffs, x):
    """Plaintext Chebyshev değerlendirmesi (Validation için)"""
    T = [np.ones_like(x), x]
    for i in range(2, len(coeffs)):
        T.append(2 * x * T[i-1] - T[i-2])

    res = np.zeros_like(x)
    for i, c in enumerate(coeffs):
        res += c * T[i]
    return res
def prepare_data():
    """Iris veriseti üzerinde Logistic Regression eğit."""
    print("[*] Iris veriseti yükleniyor ve model eğitiliyor...")
    iris = load_iris()
    X, y = iris.data, iris.target
    # Binary sınıflandırma: Setosa (0) = 1, Diğerleri = 0
    y_bin = (y == 0).astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_bin, test_size=0.2, random_state=42
    )
    clf = LogisticRegression(random_state=42)
    clf.fit(X_train, y_train)
    weights = clf.coef_[0]
    bias = clf.intercept_[0]
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    print(f"[✓] Model Eğitildi. Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")
    return X_test, y_test, weights, bias, scaler, clf
def run_bootstrapped_fhe():
    X_test, y_test, weights, bias, scaler, clf = prepare_data()
    print("\n" + "=" * 70)
    print("  Bootstrapping: Gerçek FHE Iris Sınıflandırma")
    print("=" * 70)
    # ─────────────────────────────────────────────────────────────────
    # ADIM 0: FHE Engine + Bootstrap Anahtarları
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...")
    t0 = time.time()
    # Bootstrapping-enabled motor (Parametreler kütüphane tarafından sabitlenmiştir)
    engine = Engine(use_bootstrap=True)
    print("  → Secret key üretiliyor...")
    sk = engine.create_secret_key()
    print("  → Public key üretiliyor...")
    pk = engine.create_public_key(sk)
    print("  → Relinearization key üretiliyor...")
    rk = engine.create_relinearization_key(sk)
    print("  → Conjugation key üretiliyor...")
    conj_key = engine.create_conjugation_key(sk)
    print("  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...")
    t_bk = time.time()
    lbk = engine.create_lossy_bootstrap_key(sk, stage_count=3)
    print(f"  → Bootstrap key hazır. Süre: {time.time() - t_bk:.2f} sn")
    total_setup = time.time() - t0
    print(f"\n[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: {total_setup:.2f} sn")
    print(f"    Slot sayısı: {engine.slot_count}")
    print(f"    Maksimum level: {engine.max_level}")
    # Chebyshev katsayıları
    step_coeffs = chebyshev_coeffs_step(degree=15, delta=0.05)
    # ─────────────────────────────────────────────────────────────────
    # Birden fazla test örneği üzerinde ardışık FHE inference
    # ─────────────────────────────────────────────────────────────────
    n_samples = min(5, len(X_test))
    print(f"\n{'=' * 70}")
    print(f"  {n_samples} test örneği üzerinde şifreli tahmin")
    print(f"{'=' * 70}")
    correct_count = 0
    level = engine.max_level
    for idx in range(n_samples):
        sample = X_test[idx]
        actual = y_test[idx]
        # Plaintext tahmin (doğrulama için)
        logit_pt = np.dot(sample, weights) + bias
        pred_pt = 1 if logit_pt > 0 else 0
        print(f"\n--- Örnek {idx + 1}/{n_samples} ---")
        print(f"  Özellikler: [{', '.join(f'{v:.3f}' for v in sample)}]")
        print(f"  Gerçek sınıf: {actual}  |  Plaintext tahmin: {pred_pt}")
        # ─── ADIM 1: Şifreleme ───
        t_enc = time.time()
        cts = []
        for f_val in sample:
            arr = np.zeros(engine.slot_count)
            arr[0] = f_val
            ct = engine.encrypt(arr, pk)
            cts.append(ct)
        # ─── ADIM 2: Şifreli wx + b ───
        ct_dot = None
        for i, w in enumerate(weights):
            w_arr = np.zeros(engine.slot_count)
            w_arr[0] = w
            pt_w = engine.encode(w_arr, level)
            ct_wx_i = engine.multiply(cts[i], pt_w)
            if ct_dot is None:
                ct_dot = ct_wx_i
            else:
                ct_dot = engine.add(ct_dot, ct_wx_i)
        # Bias ekleme
        b_arr = np.zeros(engine.slot_count)
        b_arr[0] = bias
        pt_b = engine.encode(b_arr, ct_dot.level)
        ct_logit = engine.add(ct_dot, pt_b)
        print(f"  Level (wx+b sonrası): {ct_logit.level}")
        # ─── ADIM 3: Bootstrap (level yenileme) ───
        # Bootstrapping'i GÖSTERMEK için eşiği yüksek tutuyoruz (25 ve altı)
        if ct_logit.level <= 25:
            print(f"  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: {ct_logit.level})...")
            t_bs = time.time()
            ct_logit_intt = engine.intt(ct_logit)
            ct_logit = engine.lossy_bootstrap(
                ct_logit_intt, rk, conj_key, lbk
            )
            print(f"  [BOOTSTRAP] Yeni level: {ct_logit.level}  (süre: {time.time() - t_bs:.2f} sn)")
        # ─── ADIM 4: Normalizasyon + BLEACH Aktivasyonu ───
        # Logit'i [-1, 1] aralığına normalize et
        scale_factor = 20.0
        s_arr = np.zeros(engine.slot_count)
        s_arr[0] = 1.0 / scale_factor
        pt_s = engine.encode(s_arr, ct_logit.level)
        ct_norm = engine.multiply(ct_logit, pt_s)
        print(f"  Level (normalize sonrası): {ct_norm.level}")
        # Level yeterliliğini kontrol et, gerekirse tekrar bootstrap
        if ct_norm.level < 5:
            print(f"  [BOOTSTRAP] Chebyshev öncesi level düşük ({ct_norm.level}), bootstrap...")
            t_bs2 = time.time()
            ct_norm_intt_bs = engine.intt(ct_norm)
            ct_norm = engine.lossy_bootstrap(
                ct_norm_intt_bs, rk, conj_key, lbk
            )
            print(f"  [BOOTSTRAP] Yeni level: {ct_norm.level}  (süre: {time.time() - t_bs2:.2f} sn)")
        # Chebyshev step fonksiyonu
        ct_norm_intt = engine.intt(ct_norm)
        ct_pred = engine.evaluate_chebyshev_polynomial(
            ct_norm_intt, step_coeffs.tolist(), rk
        )
        # ─── ADIM 5: Şifre çözme ───
        result = engine.decrypt(ct_pred, sk)
        fhe_val = result[0]
        fhe_class = int(np.round(fhe_val))
        fhe_class = max(0, min(1, fhe_class))  # 0 veya 1'e clamp
        t_total = time.time() - t_enc
        match = fhe_class == pred_pt
        if match:
            correct_count += 1
        print(f"  FHE ham değer: {fhe_val:.6f}")
        print(f"  FHE tahmin: {fhe_class} {'✓' if match else '✗'}  (süre: {t_total:.2f} sn)")
    # ─── Özet ───
    print(f"\n{'=' * 70}")
    print(f"  SONUÇ: {correct_count}/{n_samples} doğru tahmin")
    print(f"  Doğruluk: {correct_count / n_samples * 100:.1f}%")
    print(f"{'=' * 70}")
    if correct_count == n_samples:
        print("\n[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Bootstrapping başarılı.")
    else:
        print(f"\n[!] {n_samples - correct_count} tahmin uyuşmadı.")
if __name__ == "__main__":
    run_bootstrapped_fhe()

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import time
from desilofhe import Engine


# ─────────────────────────────────────────────────────────────────
# Standalone Fonksiyonları
# ─────────────────────────────────────────────────────────────────

def chebyshev_coeffs_step(degree: int, delta: float = 0.05):
    from scipy.fftpack import dct
    n = degree + 1
    x = np.cos(np.pi * (np.arange(n) + 0.5) / n)
    y = np.where(x > 0, 1.0, 0.0)
    c = dct(y, type=2, norm='ortho')
    c[0] /= np.sqrt(2)
    c *= (np.sqrt(2.0 / n))
    return c


def eval_chebyshev_plaintext(coeffs, x):
    T = [np.ones_like(x), x]
    for i in range(2, len(coeffs)):
        T.append(2 * x * T[i-1] - T[i-2])
    res = np.zeros_like(x)
    for i, c in enumerate(coeffs):
        res += c * T[i]
    return res


def prepare_data():
    print("[*] Iris veriseti yükleniyor ve model eğitiliyor...")
    iris = load_iris()
    X, y = iris.data, iris.target
    y_bin = (y == 0).astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_bin, test_size=0.2, random_state=42
    )
    clf = LogisticRegression(random_state=42)
    clf.fit(X_train, y_train)
    weights = clf.coef_[0]
    bias = clf.intercept_[0]
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    print(f"[✓] Model Eğitildi. Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")
    return X_test, y_test, weights, bias, scaler, clf


def run_bootstrapped_fhe_batched():
    X_test, y_test, weights, bias, scaler, clf = prepare_data()

    print("\n" + "=" * 70)
    print("  BLEACH + Bootstrapping: Batch Paketleme ile Optimize FHE")
    print("=" * 70)

    # ─────────────────────────────────────────────────────────────────
    # ADIM 0: FHE Engine + Bootstrap Anahtarları
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...")
    t0 = time.time()

    engine = Engine(use_bootstrap=True)
    print("  → Secret key üretiliyor...")
    sk = engine.create_secret_key()
    print("  → Public key üretiliyor...")
    pk = engine.create_public_key(sk)
    print("  → Relinearization key üretiliyor...")
    rk = engine.create_relinearization_key(sk)
    print("  → Conjugation key üretiliyor...")
    conj_key = engine.create_conjugation_key(sk)
    print("  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...")
    t_bk = time.time()
    lbk = engine.create_lossy_bootstrap_key(sk, stage_count=3)
    print(f"  → Bootstrap key hazır. Süre: {time.time() - t_bk:.2f} sn")

    total_setup = time.time() - t0
    print(f"\n[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: {total_setup:.2f} sn")
    print(f"    Slot sayısı: {engine.slot_count}")
    print(f"    Maksimum level: {engine.max_level}")

    step_coeffs = chebyshev_coeffs_step(degree=15, delta=0.05)

    n_samples = min(5, len(X_test))
    n_features = 4
    # Her örnek stride kadar slot kullanıyor, özellikler arka arkaya
    # Örnek i, özellik j → slot[i * stride + j]
    stride = n_features  # 4 — örnekler arasında boşluk yok, bitişik

    print(f"\n{'=' * 70}")
    print(f"  Paketleme: {n_samples} örnek × {n_features} özellik")
    print(f"  Toplam kullanılan slot: {n_samples * stride} / {engine.slot_count}")
    print(f"  Şifreleme sayısı: {n_features} ct (önceki: {n_samples * n_features} ct)")
    print(f"{'=' * 70}")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 1: Tüm örnekleri 4 ciphertext'e paketle
    # ─────────────────────────────────────────────────────────────────
    # ct_feat[j] → tüm örneklerin j. özelliğini içeriyor
    # ct_feat[j][slot i] = X_test[i][j]
    print("\n[ADIM 1] Batch şifreleme...")
    t_enc = time.time()

    ct_feat = []
    for j in range(n_features):
        arr = np.zeros(engine.slot_count)
        for i in range(n_samples):
            arr[i] = X_test[i][j]   # örnek i'nin j. özelliği → slot i
        ct = engine.encrypt(arr, pk)
        ct_feat.append(ct)

    print(f"  → {n_features} ciphertext ile {n_samples} örnek şifrelendi.")
    print(f"  → Şifreleme süresi: {time.time() - t_enc:.2f} sn")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 2: Şifreli wx + b — tüm örnekler paralel
    # ─────────────────────────────────────────────────────────────────
    print("\n[ADIM 2] Şifreli wx + b hesaplanıyor (tüm örnekler paralel)...")

    level = engine.max_level
    ct_dot = None

    for j, w in enumerate(weights):
        # w[j] tüm slotlara aynı değer → örnek i'nin j. özelliğiyle çarpılıyor
        w_arr = np.full(engine.slot_count, w)
        pt_w = engine.encode(w_arr, level)
        ct_wx_j = engine.multiply(ct_feat[j], pt_w)
        if ct_dot is None:
            ct_dot = ct_wx_j
        else:
            ct_dot = engine.add(ct_dot, ct_wx_j)

    # Bias: tüm slotlara aynı bias
    b_arr = np.full(engine.slot_count, bias)
    pt_b = engine.encode(b_arr, ct_dot.level)
    ct_logit = engine.add(ct_dot, pt_b)

    print(f"  → Level (wx+b sonrası): {ct_logit.level}")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 3: Bootstrap — tek seferde tüm örnekler
    # ─────────────────────────────────────────────────────────────────
    if ct_logit.level <= 25:
        print(f"\n[ADIM 3] Bootstrap (tek seferlik, tüm örnekler)...")
        print(f"  Mevcut level: {ct_logit.level}")
        t_bs = time.time()
        ct_logit_intt = engine.intt(ct_logit)
        ct_logit = engine.lossy_bootstrap(ct_logit_intt, rk, conj_key, lbk)
        print(f"  → Yeni level: {ct_logit.level}  (süre: {time.time() - t_bs:.2f} sn)")

    # ─────────────────────────────────────────────────────────────────
    # ADIM 4: Normalizasyon + aktivasyonu — tüm örnekler paralel
    # ─────────────────────────────────────────────────────────────────
    print(f"\n[ADIM 4] Normalizasyon + Chebyshev step (tüm örnekler paralel)...")

    scale_factor = 20.0
    s_arr = np.full(engine.slot_count, 1.0 / scale_factor)
    pt_s = engine.encode(s_arr, ct_logit.level)
    ct_norm = engine.multiply(ct_logit, pt_s)
    print(f"  → Level (normalize sonrası): {ct_norm.level}")

    if ct_norm.level < 5:
        print(f"  [BOOTSTRAP] Chebyshev öncesi level düşük ({ct_norm.level}), bootstrap...")
        t_bs2 = time.time()
        ct_norm_intt_bs = engine.intt(ct_norm)
        ct_norm = engine.lossy_bootstrap(ct_norm_intt_bs, rk, conj_key, lbk)
        print(f"  → Yeni level: {ct_norm.level}  (süre: {time.time() - t_bs2:.2f} sn)")

    ct_norm_intt = engine.intt(ct_norm)
    ct_pred = engine.evaluate_chebyshev_polynomial(
        ct_norm_intt, step_coeffs.tolist(), rk
    )

    # ─────────────────────────────────────────────────────────────────
    # ADIM 5: Şifre çözme — tüm örnekler tek decrypt
    # ─────────────────────────────────────────────────────────────────
    print(f"\n[ADIM 5] Şifre çözme (tek decrypt, tüm örnekler)...")
    result = engine.decrypt(ct_pred, sk)

    # ─────────────────────────────────────────────────────────────────
    # Sonuçları ayıkla ve raporla
    # ─────────────────────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print(f"  Sonuçlar")
    print(f"{'=' * 70}")

    correct_count = 0
    for i in range(n_samples):
        fhe_val = result[i]   # örnek i → slot i
        fhe_class = int(np.round(fhe_val))
        fhe_class = max(0, min(1, fhe_class))

        # Plaintext doğrulama
        logit_pt = np.dot(X_test[i], weights) + bias
        pred_pt = 1 if logit_pt > 0 else 0
        actual = y_test[i]
        match = fhe_class == pred_pt

        if match:
            correct_count += 1

        print(f"\n  Örnek {i + 1}/{n_samples}")
        print(f"    Özellikler: [{', '.join(f'{v:.3f}' for v in X_test[i])}]")
        print(f"    Gerçek sınıf: {actual}  |  Plaintext tahmin: {pred_pt}")
        print(f"    FHE ham değer: {fhe_val:.6f}")
        print(f"    FHE tahmin: {fhe_class} {'✓' if match else '✗'}")

    print(f"\n{'=' * 70}")
    print(f"  SONUÇ: {correct_count}/{n_samples} doğru tahmin")
    print(f"  Doğruluk: {correct_count / n_samples * 100:.1f}%")
    print(f"{'=' * 70}")

    if correct_count == n_samples:
        print("\n[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! SIMD bootstrapping başarılı.")
    else:
        print(f"\n[!] {n_samples - correct_count} tahmin uyuşmadı.")


if __name__ == "__main__":
    run_bootstrapped_fhe_batched()

# Output

In [ ]:
[*] Iris veriseti yükleniyor ve model eğitiliyor...
[✓] Model Eğitildi. Train acc: 1.0000, Test acc: 1.0000

======================================================================
  BLEACH + Bootstrapping: Gerçek FHE Iris Sınıflandırma
======================================================================

[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...
  → Secret key üretiliyor...
  → Public key üretiliyor...
  → Relinearization key üretiliyor...
  → Conjugation key üretiliyor...
  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...
  → Bootstrap key hazır. Süre: 100.02 sn

[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: 104.45 sn
    Slot sayısı: 32768
    Maksimum level: 26

======================================================================
  5 test örneği üzerinde şifreli tahmin
======================================================================

--- Örnek 1/5 ---
  Özellikler: [0.311, -0.592, 0.535, 0.001]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 33.05 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.126044
  FHE tahmin: 0 ✓  (süre: 41.89 sn)

--- Örnek 2/5 ---
  Özellikler: [-0.174, 1.710, -1.170, -1.184]
  Gerçek sınıf: 1  |  Plaintext tahmin: 1
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.61 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 1.141355
  FHE tahmin: 1 ✓  (süre: 41.29 sn)

--- Örnek 3/5 ---
  Özellikler: [2.250, -1.053, 1.786, 1.449]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.59 sn)
  Level (normalize sonrası): 12
  FHE ham değer: 0.021774
  FHE tahmin: 0 ✓  (süre: 41.20 sn)

--- Örnek 4/5 ---
  Özellikler: [0.190, -0.362, 0.422, 0.396]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.58 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.125001
  FHE tahmin: 0 ✓  (süre: 41.21 sn)

--- Örnek 5/5 ---
  Özellikler: [1.159, -0.592, 0.592, 0.264]
  Gerçek sınıf: 0  |  Plaintext tahmin: 0
  Level (wx+b sonrası): 25
  [BOOTSTRAP] Seviye yenileniyor (Mevcut Seviye: 25)...
  [BOOTSTRAP] Yeni level: 13  (süre: 32.60 sn)
  Level (normalize sonrası): 12
  FHE ham değer: -0.034954
  FHE tahmin: 0 ✓  (süre: 41.22 sn)

======================================================================
  SONUÇ: 5/5 doğru tahmin
  Doğruluk: 100.0%
======================================================================

[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Bootstrapping başarılı.

In [ ]:
[*] Iris veriseti yükleniyor ve model eğitiliyor...
[✓] Model Eğitildi. Train acc: 1.0000, Test acc: 1.0000

======================================================================
  BLEACH + Bootstrapping: Batch Paketleme ile Optimize FHE
======================================================================

[ADIM 0] FHE Engine (bootstrap destekli) başlatılıyor...
  → Secret key üretiliyor...
  → Public key üretiliyor...
  → Relinearization key üretiliyor...
  → Conjugation key üretiliyor...
  → Lossy Bootstrap key üretiliyor (bu biraz sürebilir)...
  → Bootstrap key hazır. Süre: 100.04 sn

[✓] Tüm anahtarlar hazır. Toplam kurulum süresi: 104.38 sn
    Slot sayısı: 32768
    Maksimum level: 26

======================================================================
  Paketleme: 5 örnek × 4 özellik
  Toplam kullanılan slot: 20 / 32768
  Şifreleme sayısı: 4 ct (önceki: 20 ct)
======================================================================

[ADIM 1] Batch şifreleme...
  → 4 ciphertext ile 5 örnek şifrelendi.
  → Şifreleme süresi: 1.40 sn

[ADIM 2] Şifreli wx + b hesaplanıyor (tüm örnekler paralel)...
  → Level (wx+b sonrası): 25

[ADIM 3] Bootstrap (tek seferlik, tüm örnekler)...
  Mevcut level: 25
  → Yeni level: 13  (süre: 32.71 sn)

[ADIM 4] Normalizasyon + Chebyshev step (tüm örnekler paralel)...
  → Level (normalize sonrası): 12

[ADIM 5] Şifre çözme (tek decrypt, tüm örnekler)...

======================================================================
  Sonuçlar
======================================================================

  Örnek 1/5
    Özellikler: [0.311, -0.592, 0.535, 0.001]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.126044
    FHE tahmin: 0 ✓

  Örnek 2/5
    Özellikler: [-0.174, 1.710, -1.170, -1.184]
    Gerçek sınıf: 1  |  Plaintext tahmin: 1
    FHE ham değer: 1.141355
    FHE tahmin: 1 ✓

  Örnek 3/5
    Özellikler: [2.250, -1.053, 1.786, 1.449]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: 0.021774
    FHE tahmin: 0 ✓

  Örnek 4/5
    Özellikler: [0.190, -0.362, 0.422, 0.396]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.125001
    FHE tahmin: 0 ✓

  Örnek 5/5
    Özellikler: [1.159, -0.592, 0.592, 0.264]
    Gerçek sınıf: 0  |  Plaintext tahmin: 0
    FHE ham değer: -0.034954
    FHE tahmin: 0 ✓

======================================================================
  SONUÇ: 5/5 doğru tahmin
  Doğruluk: 100.0%
======================================================================

[✓] TÜM ŞİFRELİ TAHMİNLER DOĞRU! Batch bootstrapping başarılı.

**Code OpenFHE**

In [ ]:
cpp_code = r"""
#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_NotSet);
    parameters.SetRingDim(1 << 13);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;
    int p    = 32;

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD CKKS — Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    // Rotasyon anahtarlari: iç çarpim toplamı için (önceki kodla aynı)
    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    // ── Veri ──────────────────────────────────────────────────────────────
    // 5 ornek x 4 ozellik
    std::vector<std::vector<double>> all_features = {
        { 0.311, -0.592,  0.535,  0.001},
        {-0.174,  1.710, -1.170, -1.184},
        { 2.250, -1.053,  1.786,  1.449},
        { 0.190, -0.362,  0.422,  0.396},
        { 1.159, -0.592,  0.592,  0.264}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0};
    std::vector<double> weights  = {-1.018, 1.173, -1.675, -1.536};
    double bias = -2.4196;

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    // LUT fonksiyonu — p=32 icin esik 16
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    // ── Plaintext doğrulama ───────────────────────────────────────────────
    std::cout << "\n  Plaintext logitler (dogrulama):" << std::endl;
    std::vector<int> pred_pt_all(n_samples);
    for (int i = 0; i < n_samples; i++) {
        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * all_features[i][j];
        pred_pt_all[i] = (logit_pt > 0.0) ? 1 : 0;
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);
        std::cout << "    Ornek " << (i+1) << ": logit=" << std::fixed << std::setprecision(4)
                  << logit_pt << (in_range ? "  [OK]" : "  [ARALIK DISI]")
                  << "  pred=" << pred_pt_all[i] << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  SIMD Batch Sifreli Cikarim" << std::endl;
    std::cout << "  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j" << std::endl;
    std::cout << "  " << n_samples << " ornek PARALEL, 1 bootstrap" << std::endl;
    std::cout << "========================================" << std::endl;

    // ── ADIM 1: SIMD Batch Sifreleme ──────────────────────────────────────
    // ct_feat[j]: tum orneklerin j. ozelligi tek bir ciphertext'e
    // slot[i] = all_features[i][j]
    auto t_enc_start = Clock::now();

    std::vector<Ciphertext<DCRTPoly>> ct_feat(n_features);
    for (int j = 0; j < n_features; j++) {
        std::vector<std::complex<double>> slot_vec(numSlots, {0.0, 0.0});
        for (int i = 0; i < n_samples; i++) {
            slot_vec[i] = {all_features[i][j], 0.0};
        }
        Plaintext ptxt = cc->MakeCKKSPackedPlaintext(slot_vec, 1, 0, nullptr, numSlots);
        ct_feat[j] = cc->Encrypt(keyPair.publicKey, ptxt);
    }

    auto t_enc_end = Clock::now();
    double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
    std::cout << "\n[1] SIMD Batch Sifreleme   : " << std::fixed << std::setprecision(2)
              << dt_enc << " ms  (" << n_features << " ct, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 2: SIMD wx — her agirlik tum slot'lara yayilir ───────────────
    // w[j] tum slot'larda ayni deger → ct_feat[j] ile carpilinca
    // slot[i] = w[j] * ornek_i_ozellik_j
    auto t_mult_start = Clock::now();

    Ciphertext<DCRTPoly> ct_dot;
    bool first = true;
    for (int j = 0; j < n_features; j++) {
        // w[j] tum slot'lara yayilmis plaintext
        std::vector<std::complex<double>> w_vec(numSlots, {weights[j], 0.0});
        Plaintext ptxt_w = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);

        auto ct_wx_j = cc->EvalMult(ct_feat[j], ptxt_w);
        cc->RescaleInPlace(ct_wx_j);

        if (first) { ct_dot = ct_wx_j; first = false; }
        else       { ct_dot = cc->EvalAdd(ct_dot, ct_wx_j); }
    }
    // ct_dot[i] = sum_j(w[j] * x[i][j]) = w^T x[i]  tum ornekler icin

    auto t_mult_end = Clock::now();
    double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
    std::cout << "[2] SIMD EvalMult+Add      : " << std::fixed << std::setprecision(2)
              << dt_mult << " ms  (tum ornekler paralel)" << std::endl;

    // ── ADIM 3: Bias — tum slot'lara ayni deger ───────────────────────────
    // bias skaler olarak tum slot'lara eklenir
    auto t_bias_start = Clock::now();
    auto ct_logit = cc->EvalAdd(ct_dot, bias);
    auto t_bias_end = Clock::now();
    double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
    std::cout << "[3] Bias ekleme (SIMD)     : " << std::fixed << std::setprecision(2)
              << dt_bias << " ms  (Level: " << ct_logit->GetLevel() << ")" << std::endl;

    // ── ADIM 4: Level hizalama ─────────────────────────────────────────────
    auto t_align_start = Clock::now();
    while (ct_logit->GetLevel() < targetLevel)
        cc->LevelReduceInPlace(ct_logit, nullptr);
    auto t_align_end = Clock::now();
    double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
    std::cout << "[4] Level hizalama         : " << std::fixed << std::setprecision(2)
              << dt_align << " ms  (Level: " << ct_logit->GetLevel() << "/" << targetLevel << ")" << std::endl;

    // ── ADIM 5: TEK Bootstrap — 5 ornek paralel ───────────────────────────
    // EvalFuncBootstrap ct_logit uzerinde calisir
    // Her slot[i] bagimsiz olarak f(logit_i) hesaplanir — SIMD
    auto t_bs_start = Clock::now();
    auto ct_result = cc->EvalFuncBootstrap(ct_logit, f_lut, p, 1);
    auto t_bs_end = Clock::now();
    double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
    std::cout << "[5] EvalFuncBootstrap(SIMD): " << std::fixed << std::setprecision(2)
              << dt_bs << " ms  (1 kez, " << n_samples << " ornek paralel)" << std::endl;

    // ── ADIM 6: Tek Decrypt — tum sonuclar ────────────────────────────────
    auto t_dec_start = Clock::now();
    Plaintext pt_result;
    cc->Decrypt(keyPair.secretKey, ct_result, &pt_result);
    pt_result->SetLength(n_samples);
    auto t_dec_end = Clock::now();
    double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
    std::cout << "[6] Sifre cozme (tek)      : " << std::fixed << std::setprecision(2)
              << dt_dec << " ms" << std::endl;

    // ── Sonuclari ayikla ──────────────────────────────────────────────────
    auto fhe_values = pt_result->GetCKKSPackedValue();

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sonuclar  (slot[i] = ornek i)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;
    double total_infer = dt_enc + dt_mult + dt_bias + dt_align + dt_bs + dt_dec;

    for (int i = 0; i < n_samples; i++) {
        double fhe_val = fhe_values[i].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt_all[i]);
        if (match) correct++;

        std::cout << "  Ornek " << (i+1) << ": "
                  << "logit=" << std::fixed << std::setprecision(4) << (bias + [&](){
                        double s = 0;
                        for (int j = 0; j < n_features; j++) s += weights[j]*all_features[i][j];
                        return s;
                     }())
                  << "  FHE=" << std::fixed << std::setprecision(6) << fhe_val
                  << "  pred=" << fhe_pred
                  << "  gercek=" << true_labels[i]
                  << (match ? "  [OK]" : "  [HATA]") << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (SIMD, " << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- SIMD Sifreleme      : " << dt_enc   << " ms  (" << n_features << " ct)" << std::endl;
    std::cout << "    -- SIMD EvalMult+Add   : " << dt_mult  << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << dt_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << dt_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << dt_bs    << " ms  (1 kez, " << n_samples << " paralel)" << std::endl;
    std::cout << "    -- Sifre cozme         : " << dt_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi amortize      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * dt_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}
"""

with open("/content/sec-ckks/src/pke/examples/iris_functional.cpp", "w") as f:
    f.write(cpp_code)

print("Dosya yazildi.")

In [ ]:
cpp_code = r"""
#include "openfhe.h"
#include <complex>
#include <iostream>
#include <iomanip>
#include <vector>
#include <chrono>
using namespace lbcrypto;
using namespace std::chrono;

using Clock = high_resolution_clock;
using Ms    = duration<double, std::milli>;

void IrisFunctionalBootstrapping() {

    CCParams<CryptoContextCKKSRNS> parameters;
    parameters.SetSecretKeyDist(SPARSE_TERNARY);
    parameters.SetSecurityLevel(HEStd_NotSet);
    parameters.SetRingDim(1 << 13);
    parameters.SetNumLargeDigits(3);
    parameters.SetKeySwitchTechnique(HYBRID);
    parameters.SetScalingModSize(45);
    parameters.SetScalingTechnique(FIXEDMANUAL);
    parameters.SetFirstModSize(46);

    std::vector<uint32_t> levelBudget = {4, 2};
    std::vector<uint32_t> bsgsDim     = {0, 0};
    int bits = 5;   // 2^5 = 32 = p  (p=16 ile logit=-11.16 sinir disina cikiyordu)
    int p    = 32;  // guvenli aralik: (-16, +16)

    usint depth = levelBudget[0] + levelBudget[1] + 1 + 5 + 4 + bits + 2;
    parameters.SetMultiplicativeDepth(depth);

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Anahtar Kurulum" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << "  p (modul)           : " << p << "  (guvenli aralik: (-16, +16))" << std::endl;
    std::cout << "  bits                : " << bits << std::endl;
    std::cout << "  depth               : " << depth << std::endl;

    auto t_setup_start = Clock::now();

    CryptoContext<DCRTPoly> cc = GenCryptoContext(parameters);
    cc->Enable(PKE);
    cc->Enable(KEYSWITCH);
    cc->Enable(LEVELEDSHE);
    cc->Enable(ADVANCEDSHE);
    cc->Enable(FHE);
    cc->Enable(FBTS);

    int numSlots = cc->GetRingDimension() / 2;
    cc->EvalFuncBootstrapSetup(levelBudget, bsgsDim, numSlots, bits);

    auto t_keygen_start = Clock::now();
    auto keyPair = cc->KeyGen();
    cc->EvalMultKeyGen(keyPair.secretKey);
    auto t_keygen_end = Clock::now();

    auto t_bskey_start = Clock::now();
    cc->EvalBootstrapKeyGen(keyPair.secretKey, numSlots);
    auto t_bskey_end = Clock::now();

    cc->EvalRotateKeyGen(keyPair.secretKey, {1, 2});

    auto t_setup_end = Clock::now();

    double t_keygen = duration_cast<Ms>(t_keygen_end - t_keygen_start).count();
    double t_bskey  = duration_cast<Ms>(t_bskey_end  - t_bskey_start).count();
    double t_setup  = duration_cast<Ms>(t_setup_end  - t_setup_start).count();

    std::cout << "  Ring boyutu (n)     : " << cc->GetRingDimension() << std::endl;
    std::cout << "  Slot sayisi         : " << numSlots << std::endl;
    std::cout << "  Carpim derinligi    : " << depth << std::endl;
    std::cout << "  Key uretimi         : " << std::fixed << std::setprecision(2) << t_keygen << " ms" << std::endl;
    std::cout << "  Bootstrap key       : " << std::fixed << std::setprecision(2) << t_bskey  << " ms" << std::endl;
    std::cout << "  Toplam kurulum      : " << std::fixed << std::setprecision(2) << t_setup  << " ms" << std::endl;

    std::vector<std::vector<double>> all_features = {
        { 0.311, -0.592,  0.535,  0.001},
        {-0.174,  1.710, -1.170, -1.184},
        { 2.250, -1.053,  1.786,  1.449},
        { 0.190, -0.362,  0.422,  0.396},
        { 1.159, -0.592,  0.592,  0.264}
    };
    std::vector<int> true_labels = {0, 1, 0, 0, 0};
    std::vector<double> weights  = {-1.018, 1.173, -1.675, -1.536};
    double bias = -2.4196;

    // p=32: z < 0 → z mod 32 ∈ (16, 32) → f=0 (sinif 0)
    //       z > 0 → z mod 32 ∈ ( 0, 16) → f=1 (sinif 1)
    auto f_lut = [](double x) -> double {
        return (x < 16.0) ? 1.0 : 0.0;
    };

    usint targetLevel = depth - levelBudget[1];

    int n_samples  = (int)all_features.size();
    int n_features = 4;

    double total_enc   = 0.0;
    double total_mult  = 0.0;
    double total_rot   = 0.0;
    double total_bias  = 0.0;
    double total_align = 0.0;
    double total_bs    = 0.0;
    double total_dec   = 0.0;
    double total_infer = 0.0;

    std::cout << "\n========================================" << std::endl;
    std::cout << "  Sifreli Cikarim  (" << n_samples << " ornek)" << std::endl;
    std::cout << "========================================" << std::endl;

    int correct = 0;

    for (int idx = 0; idx < n_samples; idx++) {

        auto t_infer_start = Clock::now();

        const auto& feat = all_features[idx];
        int y_true = true_labels[idx];

        double logit_pt = bias;
        for (int j = 0; j < n_features; j++) logit_pt += weights[j] * feat[j];
        int pred_pt = (logit_pt > 0.0) ? 1 : 0;

        // Modul kisit kontrolu
        bool in_range = (logit_pt > -16.0 && logit_pt < 16.0);

        std::cout << "\n--- Ornek " << (idx+1) << "/" << n_samples << " ---" << std::endl;
        std::cout << "  Ozellikler     : [";
        for (int j = 0; j < n_features; j++)
            std::cout << std::fixed << std::setprecision(3) << feat[j] << (j<3?", ":"");
        std::cout << "]" << std::endl;
        std::cout << "  Gercek sinif   : " << y_true << std::endl;
        std::cout << "  Plaintext logit: " << std::fixed << std::setprecision(4) << logit_pt
                  << (in_range ? "  [aralik OK]" : "  [ARALIK DISI - HATA BEKLENIR]") << std::endl;
        std::cout << "  Plaintext pred : " << pred_pt << std::endl;

        auto t_enc_start = Clock::now();
        std::vector<std::complex<double>> f_vec(numSlots, 0);
        for (int j = 0; j < n_features; j++) f_vec[j] = {feat[j], 0.0};
        Plaintext ptxtF = cc->MakeCKKSPackedPlaintext(f_vec, 1, 0, nullptr, numSlots);
        auto ctxtFeatures = cc->Encrypt(keyPair.publicKey, ptxtF);
        auto t_enc_end = Clock::now();
        double dt_enc = duration_cast<Ms>(t_enc_end - t_enc_start).count();
        total_enc += dt_enc;
        std::cout << "  [1] Sifreleme        : " << std::fixed << std::setprecision(2) << dt_enc << " ms" << std::endl;

        auto t_mult_start = Clock::now();
        std::vector<std::complex<double>> w_vec(numSlots, 0);
        for (int j = 0; j < n_features; j++) w_vec[j] = {weights[j], 0.0};
        Plaintext ptxtW = cc->MakeCKKSPackedPlaintext(w_vec, 1, 0, nullptr, numSlots);
        auto ctxtMult = cc->EvalMult(ctxtFeatures, ptxtW);
        cc->RescaleInPlace(ctxtMult);
        auto t_mult_end = Clock::now();
        double dt_mult = duration_cast<Ms>(t_mult_end - t_mult_start).count();
        total_mult += dt_mult;
        std::cout << "  [2] EvalMult+Rescale : " << std::fixed << std::setprecision(2) << dt_mult << " ms" << std::endl;

        auto t_rot_start = Clock::now();
        auto ctxtSum = ctxtMult;
        for (int i = 0; i < 2; i++)
            ctxtSum = cc->EvalAdd(ctxtSum, cc->EvalAtIndex(ctxtSum, 1 << i));
        auto t_rot_end = Clock::now();
        double dt_rot = duration_cast<Ms>(t_rot_end - t_rot_start).count();
        total_rot += dt_rot;
        std::cout << "  [3] Rotasyon toplami : " << std::fixed << std::setprecision(2) << dt_rot << " ms" << std::endl;

        auto t_bias_start = Clock::now();
        auto ctxtLogit = cc->EvalAdd(ctxtSum, bias);
        auto t_bias_end = Clock::now();
        double dt_bias = duration_cast<Ms>(t_bias_end - t_bias_start).count();
        total_bias += dt_bias;
        std::cout << "  [4] Bias ekleme      : " << std::fixed << std::setprecision(2) << dt_bias << " ms"
                  << "  (Level: " << ctxtLogit->GetLevel() << ")" << std::endl;

        auto t_align_start = Clock::now();
        while (ctxtLogit->GetLevel() < targetLevel)
            cc->LevelReduceInPlace(ctxtLogit, nullptr);
        auto t_align_end = Clock::now();
        double dt_align = duration_cast<Ms>(t_align_end - t_align_start).count();
        total_align += dt_align;
        std::cout << "  [5] Level hizalama   : " << std::fixed << std::setprecision(2) << dt_align << " ms"
                  << "  (Level: " << ctxtLogit->GetLevel() << "/" << targetLevel << ")" << std::endl;

        auto t_bs_start = Clock::now();
        auto ctxtResult = cc->EvalFuncBootstrap(ctxtLogit, f_lut, p, 1);
        auto t_bs_end = Clock::now();
        double dt_bs = duration_cast<Ms>(t_bs_end - t_bs_start).count();
        total_bs += dt_bs;
        std::cout << "  [6] EvalFuncBootstrap: " << std::fixed << std::setprecision(2) << dt_bs << " ms" << std::endl;

        auto t_dec_start = Clock::now();
        Plaintext result;
        cc->Decrypt(keyPair.secretKey, ctxtResult, &result);
        result->SetLength(1);
        auto t_dec_end = Clock::now();
        double dt_dec = duration_cast<Ms>(t_dec_end - t_dec_start).count();
        total_dec += dt_dec;

        double fhe_val = result->GetCKKSPackedValue()[0].real();
        int fhe_pred   = (fhe_val > 0.5) ? 1 : 0;
        bool match     = (fhe_pred == pred_pt);
        if (match) correct++;

        auto t_infer_end = Clock::now();
        double dt_infer = duration_cast<Ms>(t_infer_end - t_infer_start).count();
        total_infer += dt_infer;

        std::cout << "  [7] Sifre cozme      : " << std::fixed << std::setprecision(2) << dt_dec << " ms" << std::endl;
        std::cout << "  --------------------------------" << std::endl;
        std::cout << "  FHE ham deger   : " << std::fixed << std::setprecision(6) << fhe_val << std::endl;
        std::cout << "  FHE tahmin      : " << fhe_pred << (match ? "  [OK]" : "  [HATA]") << std::endl;
        std::cout << "  Ornek suresi    : " << std::fixed << std::setprecision(2) << dt_infer << " ms" << std::endl;
    }

    std::cout << "\n========================================" << std::endl;
    std::cout << "  HESAPLAMA PROFILI  (" << n_samples << " ornek toplam)" << std::endl;
    std::cout << "========================================" << std::endl;
    std::cout << std::fixed << std::setprecision(2);
    std::cout << "  Anahtar kurulum (toplam) : " << t_setup   << " ms" << std::endl;
    std::cout << "    -- Key uretimi         : " << t_keygen  << " ms" << std::endl;
    std::cout << "    -- Bootstrap key       : " << t_bskey   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Sifreli cikarim (toplam) : " << total_infer << " ms" << std::endl;
    std::cout << "    -- Sifreleme           : " << total_enc   << " ms" << std::endl;
    std::cout << "    -- EvalMult + Rescale  : " << total_mult  << " ms" << std::endl;
    std::cout << "    -- Rotasyon toplami    : " << total_rot   << " ms" << std::endl;
    std::cout << "    -- Bias ekleme         : " << total_bias  << " ms" << std::endl;
    std::cout << "    -- Level hizalama      : " << total_align << " ms" << std::endl;
    std::cout << "    -- EvalFuncBootstrap   : " << total_bs    << " ms" << std::endl;
    std::cout << "    -- Sifre cozme         : " << total_dec   << " ms" << std::endl;
    std::cout << "  --------------------------------" << std::endl;
    std::cout << "  Ornek basi ortalama      : " << (total_infer / n_samples) << " ms" << std::endl;
    std::cout << "  Bootstrap orani          : " << (100.0 * total_bs / total_infer) << " %" << std::endl;
    std::cout << "  ================================" << std::endl;
    std::cout << "  SONUC: " << correct << "/" << n_samples
              << "  Dogruluk: " << (100.0 * correct / n_samples) << " %" << std::endl;
    std::cout << "========================================\n" << std::endl;
}

int main() {
    IrisFunctionalBootstrapping();
    return 0;
}
"""

with open("/content/sec-ckks/src/pke/examples/iris_functional.cpp", "w") as f:
    f.write(cpp_code)

print("Dosya yazildi.")

**Output**

In [ ]:
/content/sec-ckks
[ 96%] Built target pkeobj
[ 96%] Built target OPENFHEpke
[ 96%] Building CXX object src/pke/CMakeFiles/iris_functional.dir/examples/iris_functional.cpp.o
[100%] Linking CXX executable ../../bin/examples/pke/iris_functional
[100%] Built target iris_functional

========================================
  Anahtar Kurulum
========================================
  p (modul)           : 32  (guvenli aralik: (-16, +16))
  bits                : 5
  depth               : 23
  Ring boyutu (n)     : 8192
  Slot sayisi         : 4096
  Carpim derinligi    : 23
  Key uretimi         : 136.55 ms
  Bootstrap key       : 1601.04 ms
  Toplam kurulum      : 2496.27 ms

========================================
  Sifreli Cikarim  (5 ornek)
========================================

--- Ornek 1/5 ---
  Ozellikler     : [0.311, -0.592, 0.535, 0.001]
  Gercek sinif   : 0
  Plaintext logit: -4.3283  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 23.56 ms
  [2] EvalMult+Rescale : 9.35 ms
  [3] Rotasyon toplami : 67.87 ms
  [4] Bias ekleme      : 0.52 ms  (Level: 1)
  [5] Level hizalama   : 0.13 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2535.70 ms
  [7] Sifre cozme      : 5.17 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 2642.50 ms

--- Ornek 2/5 ---
  Ozellikler     : [-0.174, 1.710, -1.170, -1.184]
  Gercek sinif   : 1
  Plaintext logit: 3.5417  [aralik OK]
  Plaintext pred : 1
  [1] Sifreleme        : 24.58 ms
  [2] EvalMult+Rescale : 8.54 ms
  [3] Rotasyon toplami : 52.28 ms
  [4] Bias ekleme      : 0.49 ms  (Level: 1)
  [5] Level hizalama   : 0.09 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2378.83 ms
  [7] Sifre cozme      : 5.08 ms
  --------------------------------
  FHE ham deger   : 1.000000
  FHE tahmin      : 1  [OK]
  Ornek suresi    : 2470.05 ms

--- Ornek 3/5 ---
  Ozellikler     : [2.250, -1.053, 1.786, 1.449]
  Gercek sinif   : 0
  Plaintext logit: -11.1625  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 24.49 ms
  [2] EvalMult+Rescale : 8.34 ms
  [3] Rotasyon toplami : 53.03 ms
  [4] Bias ekleme      : 0.49 ms  (Level: 1)
  [5] Level hizalama   : 0.09 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2412.48 ms
  [7] Sifre cozme      : 4.92 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 2504.00 ms

--- Ornek 4/5 ---
  Ozellikler     : [0.190, -0.362, 0.422, 0.396]
  Gercek sinif   : 0
  Plaintext logit: -4.3528  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 24.88 ms
  [2] EvalMult+Rescale : 8.59 ms
  [3] Rotasyon toplami : 53.79 ms
  [4] Bias ekleme      : 0.49 ms  (Level: 1)
  [5] Level hizalama   : 0.09 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 3801.00 ms
  [7] Sifre cozme      : 5.14 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 3894.17 ms

--- Ornek 5/5 ---
  Ozellikler     : [1.159, -0.592, 0.592, 0.264]
  Gercek sinif   : 0
  Plaintext logit: -5.6910  [aralik OK]
  Plaintext pred : 0
  [1] Sifreleme        : 24.66 ms
  [2] EvalMult+Rescale : 8.47 ms
  [3] Rotasyon toplami : 72.73 ms
  [4] Bias ekleme      : 5.99 ms  (Level: 1)
  [5] Level hizalama   : 0.10 ms  (Level: 21/21)
  [6] EvalFuncBootstrap: 2424.53 ms
  [7] Sifre cozme      : 4.78 ms
  --------------------------------
  FHE ham deger   : 0.000000
  FHE tahmin      : 0  [OK]
  Ornek suresi    : 2541.43 ms

========================================
  HESAPLAMA PROFILI  (5 ornek toplam)
========================================
  Anahtar kurulum (toplam) : 2496.27 ms
    -- Key uretimi         : 136.55 ms
    -- Bootstrap key       : 1601.04 ms
  --------------------------------
  Sifreli cikarim (toplam) : 14052.15 ms
    -- Sifreleme           : 122.17 ms
    -- EvalMult + Rescale  : 43.29 ms
    -- Rotasyon toplami    : 299.69 ms
    -- Bias ekleme         : 7.97 ms
    -- Level hizalama      : 0.49 ms
    -- EvalFuncBootstrap   : 13552.54 ms
    -- Sifre cozme         : 25.08 ms
  --------------------------------
  Ornek basi ortalama      : 2810.43 ms
  Bootstrap orani          : 96.44 %
  ================================
  SONUC: 5/5  Dogruluk: 100.00 %
========================================



In [ ]:
/content/sec-ckks
[ 96%] Built target pkeobj
[ 96%] Built target OPENFHEpke
[ 96%] Building CXX object src/pke/CMakeFiles/iris_functional.dir/examples/iris_functional.cpp.o
[100%] Linking CXX executable ../../bin/examples/pke/iris_functional
[100%] Built target iris_functional

========================================
  SIMD CKKS — Anahtar Kurulum
========================================
  p (modul)           : 32  (guvenli aralik: (-16, +16))
  bits                : 5
  depth               : 23
  Ring boyutu (n)     : 8192
  Slot sayisi         : 4096
  Carpim derinligi    : 23
  Key uretimi         : 136.94 ms
  Bootstrap key       : 1580.99 ms
  Toplam kurulum      : 2480.35 ms

  Plaintext logitler (dogrulama):
    Ornek 1: logit=-4.3283  [OK]  pred=0
    Ornek 2: logit=3.5417  [OK]  pred=1
    Ornek 3: logit=-11.1625  [OK]  pred=0
    Ornek 4: logit=-4.3528  [OK]  pred=0
    Ornek 5: logit=-5.6910  [OK]  pred=0

========================================
  SIMD Batch Sifreli Cikarim
  Strateji: ct_feat[j][slot_i] = ornek_i ozellik_j
  5 ornek PARALEL, 1 bootstrap
========================================

[1] SIMD Batch Sifreleme   : 110.99 ms  (4 ct, 5 ornek paralel)
[2] SIMD EvalMult+Add      : 43.10 ms  (tum ornekler paralel)
[3] Bias ekleme (SIMD)     : 0.53 ms  (Level: 1)
[4] Level hizalama         : 0.09 ms  (Level: 21/21)
[5] EvalFuncBootstrap(SIMD): 2462.26 ms  (1 kez, 5 ornek paralel)
[6] Sifre cozme (tek)      : 4.83 ms

========================================
  Sonuclar  (slot[i] = ornek i)
========================================
  Ornek 1: logit=-4.3283  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 2: logit=3.5417  FHE=1.000000  pred=1  gercek=1  [OK]
  Ornek 3: logit=-11.1625  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 4: logit=-4.3528  FHE=0.000000  pred=0  gercek=0  [OK]
  Ornek 5: logit=-5.6910  FHE=0.000000  pred=0  gercek=0  [OK]

========================================
  HESAPLAMA PROFILI  (SIMD, 5 ornek)
========================================
  Anahtar kurulum (toplam) : 2480.35 ms
    -- Key uretimi         : 136.94 ms
    -- Bootstrap key       : 1580.99 ms
  --------------------------------
  Sifreli cikarim (toplam) : 2621.79 ms
    -- SIMD Sifreleme      : 110.99 ms  (4 ct)
    -- SIMD EvalMult+Add   : 43.10 ms
    -- Bias ekleme         : 0.53 ms
    -- Level hizalama      : 0.09 ms
    -- EvalFuncBootstrap   : 2462.26 ms  (1 kez, 5 paralel)
    -- Sifre cozme         : 4.83 ms
  --------------------------------
  Ornek basi amortize      : 524.36 ms
  Bootstrap orani          : 93.92 %
  ================================
  SONUC: 5/5  Dogruluk: 100.00 %
========================================